# Thinking-aware pilot: LoRA SFT

First experiment for the thinking-aware instruction-tuning direction (see `CLAUDE.md` / `PLAN.md` in the external data repo).

- **Base model:** [`khairi/Eshmun-Thinking-Pilot`](https://huggingface.co/khairi/Eshmun-Thinking-Pilot) — InstructProtein (OPT-1.3B) with `<think>`/`</think>` added as single-unit tokens and the embedding matrix resized/warm-started.
- **Dataset:** [`khairi/eshmun-thinking-pilot`](https://huggingface.co/datasets/khairi/eshmun-thinking-pilot) — 2200 rows (1200 annotation + 1000 generation), columns `Entry`, `Instruction`, `Answer`, `Reasoning`.
- **Method:** LoRA (rank 64, alpha 128, all attention projections — the project's established recipe, see `docs/paper/eshmun.md`), via TRL's `SFTTrainer`.

Each row is reshaped into a TRL prompt-completion pair:
```
prompt     = Instruction
completion = "<think>\n{Reasoning}\n</think>\n{Answer}"
```
With this shape, TRL defaults to **completion-only loss** (no gradient on the instruction tokens) and **automatically appends `tokenizer.eos_token` to the end of every completion** during dataset preparation.

This notebook mirrors `scripts/sft/train_thinking_pilot.py` for interactive use — same logic, cell by cell.

Ready to run end to end (section 7 launches training). `DATASET_CONFIGS` in section 2 selects T1 (thinking, default) vs. B1 (non-thinking ablation) -- see `docs/thinking_pilot_evaluation_protocol.md` for what the two conditions are for.

## 0. Install dependencies (Colab)

Colab's default image doesn't have `peft`, `bitsandbytes`, or a `trl` recent enough for the API used below (`SFTConfig.max_length`, prompt-completion auto-detection).

`trl` and `peft` are upper-bounded deliberately, not just floor-pinned: `trl>=1.0` breaks LoRA + `trainable_token_indices` on a tied `lm_head`/`embed_tokens` pair. Its new chunked cross-entropy path (`sft_trainer.py::_chunked_ce_forward`) does `lm_head.bias` with no `getattr` fallback, and PEFT's `TrainableTokensWrapper` (what wraps `lm_head` here, via section 5's `trainable_token_indices`) only forwards a few known attributes -- not `.bias` -- so it raises `AttributeError: 'TrainableTokensWrapper' object has no attribute 'bias'`. Verified by reproducing locally: `trl==0.29.1` works, `trl==1.8.0` fails with this exact trace, regardless of `gradient_checkpointing`.

In [ ]:
!pip install -q "transformers>=5.8.1" "torchao>=0.16.0" "datasets>=3.1.0" "accelerate>=1.12.0" "trl>=0.29.1,<1.0" "peft>=0.18.1,<0.19" "bitsandbytes>=0.49.1"

## 1. Imports

In [ ]:
import torch
from datasets import concatenate_datasets, load_dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

## 2. Config

`LORA_TARGET_MODULES` covers the attention projections only, per the project's established LoRA recipe.

`NEW_TOKENS` matters specifically for this checkpoint: `embed_tokens`/`lm_head` are an `nn.Embedding`/`nn.Linear` pair, not attention projections, so `LORA_TARGET_MODULES` never touches them and they'd stay frozen by default. That would permanently strand the two newly added `<think>`/`</think>` embeddings at their untrained, mean-init starting value with zero gradient. The LoRA config (section 5) uses `trainable_token_indices={"embed_tokens": [...]}` to train only those two embedding rows -- see [PEFT's Trainable Tokens docs](https://huggingface.co/docs/peft/package_reference/trainable_tokens) and the [`LoraConfig` reference](https://huggingface.co/docs/peft/package_reference/lora). This is more targeted than `modules_to_save`, which would make the *entire* 50304x2048 embedding matrix trainable (every pretrained row, not just the 2 new ones). PEFT auto-detects that `embed_tokens`/`lm_head` are tied on this checkpoint (`model._tied_weights_keys`, `config.tie_word_embeddings=True` -- verified directly against this checkpoint) and applies a linked adapter to `lm_head` too, so listing `embed_tokens` alone is enough.

In [ ]:
BASE_MODEL = "khairi/Eshmun-Thinking-Pilot"
DATASET = "khairi/eshmun-thinking-pilot"
# T1 (thinking): ["annotation_thinking", "generation_thinking"]
# B1 (direct-SFT ablation): ["annotation_non_thinking", "generation_non_thinking"]
DATASET_CONFIGS = ["annotation_thinking", "generation_thinking"]
EVAL_SPLIT = "validation"  # set to "" to skip eval during training

# Derived from DATASET_CONFIGS so flipping to the B1 ablation (just change the line
# above to ["annotation_non_thinking", "generation_non_thinking"]) also renames the
# output/push target -- no separate manual edit to forget, and it can never silently
# overwrite the other condition's checkpoint or the pre-split khairi/eshmun-thinking-pilot-lora.
CONDITION = "b1" if "non_thinking" in DATASET_CONFIGS[0] else "t1"
OUTPUT_DIR = f"checkpoints/eshmun-thinking-pilot-{CONDITION}"
HUB_REPO = f"khairi/eshmun-thinking-pilot-{CONDITION}-lora"

DTYPE = "float32"  # "float32" | "float16"
DTYPE_MAP = {"float32": torch.float32, "float16": torch.float16}

MAX_LENGTH = 2048

LORA_R = 64
LORA_ALPHA = 128
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "out_proj"]
NEW_TOKENS = ["<think>", "</think>"]

LOAD_IN_4BIT = False

NUM_EPOCHS = 3.0
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 16
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.03
SEED = 42

## 3. Load tokenizer + base model

Loads in `DTYPE` (configured above; defaults to `float32` per project convention, run on Colab which has the VRAM headroom this 1.3B-parameter base model needs in full precision). Switch `DTYPE` to `"float16"` in the config cell if you need the smaller footprint -- see the note there about float16's narrower numeric range.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

torch_dtype = DTYPE_MAP[DTYPE]

quantization_config = None
if LOAD_IN_4BIT:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch_dtype,
        bnb_4bit_use_double_quant=True,
    )

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=torch_dtype,
    quantization_config=quantization_config,
)
model.config.use_cache = False
model.enable_input_require_grads()

## 4. Load + prepare the dataset

Reshape `(Entry, Instruction, Answer, [Reasoning])` into `(prompt, completion)`, then drop examples whose tokenized `prompt + completion` exceeds `MAX_LENGTH` — the pilot-quality check found ~2% of examples exceed this checkpoint's 2048-token context window, and truncating from the right (TRL's default) would cut off the `<think>` block or the answer rather than the instruction.

`DATASET_CONFIGS` selects the ablation condition (T1 thinking vs. B1 non-thinking, see section 2) -- `build_example` handles both: rows without a `Reasoning` column (the non-thinking configs) collapse to an answer-only completion. `EVAL_SPLIT` loads the held-out validation split (see `docs/thinking_pilot_evaluation_protocol.md` and `scripts/python/thinking/split_annotation_by_identity.py` / `split_generation_random.py` in the external data repo for how train/validation/test were built) so training reports real held-out loss instead of only train loss.

In [ ]:
def build_example(example: dict) -> dict:
    reasoning = example.get("Reasoning")
    completion = (
        f"<think>\n{reasoning}\n</think>\n{example['Answer']}"
        if reasoning is not None
        else example["Answer"]
    )
    return {"prompt": example["Instruction"], "completion": completion}


def load_split(split: str):
    parts = [load_dataset(DATASET, config, split=split) for config in DATASET_CONFIGS]
    return concatenate_datasets(parts)


def fits_context(example: dict) -> bool:
    full_text = example["prompt"] + example["completion"]
    return len(tokenizer(full_text)["input_ids"]) <= MAX_LENGTH


def prepare(split: str):
    ds = load_split(split).shuffle(seed=SEED)
    ds = ds.map(build_example, remove_columns=ds.column_names)
    before = len(ds)
    ds = ds.filter(fits_context)
    print(f"[{split}] dropped {before - len(ds)}/{before} examples exceeding max_length={MAX_LENGTH}")
    return ds


dataset = prepare("train")
eval_dataset = prepare(EVAL_SPLIT) if EVAL_SPLIT else None

## 5. LoRA config

In [ ]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    trainable_token_indices={"embed_tokens": tokenizer.convert_tokens_to_ids(NEW_TOKENS)},
    task_type="CAUSAL_LM",
    bias="none",
)

## 6. Training arguments + trainer

`eos_token=None` -> defaults to `tokenizer.eos_token` (`"</s>"`); TRL appends it to the end of every `completion` when preparing a prompt-completion dataset.

`completion_only_loss=None` -> defaults to `True` for prompt-completion datasets: loss is computed only on `"<think>...</think>\n{answer}"`, not the instruction.

In [ ]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    max_length=MAX_LENGTH,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    gradient_checkpointing=True,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch" if eval_dataset is not None else "no",
    report_to="none",
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)

## 6b. Confirm LoRA was actually applied

`SFTTrainer(peft_config=lora_config, ...)` wraps the model internally (`get_peft_model(model, peft_config)`, TRL's own code) -- but that reassignment happens to TRL's *internal* copy, not to this notebook's `model` variable from section 3, which still refers to the bare, un-wrapped base model. The trainable, LoRA-wrapped model only exists as `trainer.model`.

Re-pointing `model` at `trainer.model` here so the rest of the notebook (and manual inspection) uses the actual model being trained, and printing the trainable-parameter count as a concrete check that LoRA + `trainable_token_indices` took effect (attention adapters and just the two `<think>`/`</think>` embedding rows should show as trainable; everything else, including the rest of the embedding matrix, frozen).

In [ ]:
model = trainer.model
model.print_trainable_parameters()

## 7. Train

Launches training for whichever condition `DATASET_CONFIGS` (section 2) is currently set to, saves the adapter to `OUTPUT_DIR`, and pushes it to `HUB_REPO` (`khairi/eshmun-thinking-pilot-t1-lora` or `-b1-lora`, not the pre-split `khairi/eshmun-thinking-pilot-lora` from the earlier sanity-check run).

In [ ]:
trainer.train()
trainer.save_model(OUTPUT_DIR)
trainer.push_to_hub(HUB_REPO)
print(f"LoRA adapter saved to {OUTPUT_DIR} and pushed to {HUB_REPO}")